In [1]:

import numpy as np
import gurobipy as gp
from gurobipy import GRB

# =====================================================
# DADOS DO PROBLEMA
# =====================================================

v = np.array([0.019967, 0.036633, 0.056633])

Q = np.array([
    [0.00006667,  0.00009989,  0.00000122],
    [0.00009989,  0.00015534, -0.00004300],
    [0.00000122, -0.00004300,  0.00035467]
])

rho = 0.04
alpha = 2
n = len(v)

ativos = ["Ativo 1", "Ativo 2", "Ativo 3"]


# =====================================================
# FUNÇÃO PARA RESOLVER CADA SUBPROBLEMA
# =====================================================

def resolver_no(nome, fixacoes=None, relaxacao=False):

    if fixacoes is None:
        fixacoes = {}

    model = gp.Model(nome)
    model.Params.OutputFlag = 0

    # Variáveis x
    x = model.addMVar(
        n,
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="x"
    )

    # Variáveis y
    if relaxacao:
        y = model.addMVar(
            n,
            lb=0,
            ub=1,
            vtype=GRB.CONTINUOUS,
            name="y"
        )
    else:
        y = model.addMVar(
            n,
            vtype=GRB.BINARY,
            name="y"
        )

    # Função objetivo
    model.setObjective(x @ Q @ x, GRB.MINIMIZE)

    # Restrições
    model.addConstr(v @ x >= rho, "retorno_minimo")
    model.addConstr(x.sum() == 1, "capital")
    model.addConstr(x <= y, "link_x_y")
    model.addConstr(y.sum() <= alpha, "cardinalidade")

    # Fixar variáveis conforme a ramificação
    for indice, valor in fixacoes.items():
        model.addConstr(y[indice] == valor)

    # Resolver
    model.optimize()

    print("\n" + "=" * 55)
    print(f"SUBPROBLEMA {nome}")
    print("=" * 55)

    if model.status == GRB.OPTIMAL:

        pesos = x.X
        variaveis_y = y.X

        retorno = v @ pesos
        risco = pesos @ Q @ pesos

        selecionados = np.where(pesos > 1e-6)[0]

        print("Status: ÓTIMO")
        print(f"Valor da função objetivo: {risco:.10f}")
        print(f"Retorno da carteira: {retorno:.8f}")
        print(f"Soma dos pesos: {pesos.sum():.8f}")
        print(f"Quantidade de ativos com peso positivo: {len(selecionados)}")

        print("\nPesos dos ativos:")

        for i in range(n):
            print(
                f"{ativos[i]}: "
                f"x = {pesos[i]:.6f} | "
                f"y = {variaveis_y[i]:.6f}"
            )

        # Verificação das restrições
        print("\nValidação das restrições:")

        print("Retorno mínimo:",
              retorno >= rho - 1e-6)

        print("Capital total:",
              np.isclose(pesos.sum(), 1, atol=1e-6))

        print("Link x <= y:",
              np.all(pesos <= variaveis_y + 1e-6))

        print("Cardinalidade:",
              variaveis_y.sum() <= alpha + 1e-6)

        return {
            "status": "ÓTIMO",
            "objetivo": risco,
            "retorno": retorno,
            "pesos": pesos,
            "y": variaveis_y
        }

    elif model.status == GRB.INFEASIBLE:

        print("Status: INFACTÍVEL")
        return {"status": "INFACTÍVEL"}

    else:

        print(f"Status: {model.status}")
        return {"status": "OUTRO"}


# =====================================================
# RESOLUÇÃO DOS NÓS DA ÁRVORE
# =====================================================

# P1 - Relaxação contínua do problema original
P1 = resolver_no(
    "P1 - Relaxacao continua",
    relaxacao=True
)

# P2 - Ramificação y1 = 0
P2 = resolver_no(
    "P2 - y1 = 0",
    fixacoes={0: 0}
)

# P4 - Ramificação y1 = 1 e y2 = 0
# Relaxação contínua para obter o limite inferior
P4 = resolver_no(
    "P4 - y1 = 1, y2 = 0",
    fixacoes={0: 1, 1: 0},
    relaxacao=True
)

# P5 - Ramificação y1 = 1 e y2 = 1
P5 = resolver_no(
    "P5 - y1 = 1, y2 = 1",
    fixacoes={0: 1, 1: 1}
)


# =====================================================
# COMPARAÇÃO DOS RESULTADOS
# =====================================================

print("\n\n========== RESUMO DA ÁRVORE ==========")

for nome, resultado in [
    ("P1", P1),
    ("P2", P2),
    ("P4", P4),
    ("P5", P5)
]:

    if resultado["status"] == "ÓTIMO":
        print(
            f"{nome}: "
            f"FO = {resultado['objetivo']:.10f} | "
            f"Retorno = {resultado['retorno']:.6f}"
        )
    else:
        print(f"{nome}: {resultado['status']}")


# =====================================================
# VERIFICAÇÃO DA PODA POR LIMITE
# =====================================================

if P2["status"] == "ÓTIMO" and P4["status"] == "ÓTIMO":

    incumbente = P2["objetivo"]
    limite_P4 = P4["objetivo"]

    print("\n========== PODA POR LIMITE ==========")

    print(f"Incumbente P2: {incumbente:.10f}")
    print(f"Limite inferior P4: {limite_P4:.10f}")

    if limite_P4 >= incumbente - 1e-9:
        print("P4 pode ser podado por limite.")
    else:
        print("P4 não pode ser podado por limite.")


# =====================================================
# VERIFICAÇÃO DA INFAC TIBILIDADE DE P5
# =====================================================

print("\n========== VERIFICAÇÃO P5 ==========")

retorno_maximo_P5 = max(v[0], v[1])

print(f"Retorno máximo possível: {retorno_maximo_P5:.6f}")
print(f"Retorno mínimo exigido: {rho:.6f}")

if retorno_maximo_P5 < rho:
    print("P5 é infactível por não atingir o retorno mínimo.")


Set parameter Username
Set parameter LicenseID to value 2866765
Academic license - for non-commercial use only - expires 2027-09-22

SUBPROBLEMA P1 - Relaxacao continua
Status: ÓTIMO
Valor da função objetivo: 0.0000814048
Retorno da carteira: 0.04000001
Soma dos pesos: 1.00000000
Quantidade de ativos com peso positivo: 3

Pesos dos ativos:
Ativo 1: x = 0.199825 | y = 0.199825
Ativo 2: x = 0.465310 | y = 0.465310
Ativo 3: x = 0.334865 | y = 0.334865

Validação das restrições:
Retorno mínimo: True
Capital total: True
Link x <= y: True
Cardinalidade: True

SUBPROBLEMA P2 - y1 = 0
Status: ÓTIMO
Valor da função objetivo: 0.0000893365
Retorno da carteira: 0.04328859
Soma dos pesos: 1.00000000
Quantidade de ativos com peso positivo: 2

Pesos dos ativos:
Ativo 1: x = 0.000000 | y = -0.000000
Ativo 2: x = 0.667220 | y = 1.000000
Ativo 3: x = 0.332780 | y = 1.000000

Validação das restrições:
Retorno mínimo: True
Capital total: True
Link x <= y: True
Cardinalidade: True

SUBPROBLEMA P4 - y1 = 1,